In [15]:
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# ── 1) Load pkl payload (true per-participant segment embeddings + all labels) ─────
with open("deep-prep-ai-audio-embeddings.pkl", "rb") as f:
    payload = pickle.load(f)

emb_list_raw = payload["transcript_embeddings"]
prosody_list_raw = payload["prosody_features"]

LABEL_COLS = [
    "interview_score",
    "overall_personality",
    "answer_score",
    "speaking_skills",
    "agreeableness",
    "conscientiousness",
    "neuroticism",
    "openness",
    "confidence_score",
]

def to_float32_label(col_values):
    return pd.to_numeric(pd.Series(col_values), errors="coerce").to_numpy(dtype=np.float32)

Y = np.stack([to_float32_label(payload[c]) for c in LABEL_COLS], axis=1)  # (N, num_targets)


def parse_embedding_entry(entry):
    if isinstance(entry, np.ndarray):
        arr = entry.astype(np.float32, copy=False)
    elif isinstance(entry, (list, tuple)):
        arr = np.array(entry, dtype=np.float32)
    elif isinstance(entry, str):
        cleaned = (
            entry.replace("\n", " ")
            .replace("[", " ")
            .replace("]", " ")
            .replace(",", " ")
        )
        flat = np.fromstring(cleaned, sep=" ", dtype=np.float32)
        if flat.size == 0:
            return None
        arr = flat
    else:
        return None

    if arr.ndim == 0:
        return None
    if arr.ndim > 2:
        return None
    return arr


parsed = [parse_embedding_entry(entry) for entry in emb_list_raw]

# Infer embedding dimension from valid 2D entries; fallback to MiniLM dim.
embed_dim_candidates = [arr.shape[1] for arr in parsed if isinstance(arr, np.ndarray) and arr.ndim == 2 and arr.shape[1] > 0]
embed_dim = int(embed_dim_candidates[0]) if len(embed_dim_candidates) > 0 else 384

# Convert any 1D parsed vectors into (n_segments, embed_dim) if possible.
emb_list = []
for arr in parsed:
    if arr is None:
        emb_list.append(None)
        continue

    if arr.ndim == 2:
        if arr.shape[1] != embed_dim:
            emb_list.append(None)
        else:
            emb_list.append(arr.astype(np.float32, copy=False))
    else:
        if arr.size % embed_dim != 0:
            emb_list.append(None)
        else:
            emb_list.append(arr.reshape(-1, embed_dim).astype(np.float32, copy=False))

N = min(len(emb_list), len(Y), len(prosody_list_raw))
if N == 0:
    raise ValueError("No rows found in embeddings/labels.")

emb_list = emb_list[:N]
Y = Y[:N]
prosody_list_raw = prosody_list_raw[:N]

MAX_SEQ_LEN = 64
num_targets = Y.shape[1]

# X_text / X_prosody: aligned segment sequences; shared padding mask
PROSODY_DIM = 13
X = np.zeros((N, MAX_SEQ_LEN, embed_dim), dtype=np.float32)
X_prosody = np.zeros((N, MAX_SEQ_LEN, PROSODY_DIM), dtype=np.float32)
key_padding_mask = np.ones((N, MAX_SEQ_LEN), dtype=bool)

valid_modal_rows = np.zeros(N, dtype=bool)
for i, arr in enumerate(emb_list):
    if arr is None or arr.ndim != 2 or arr.shape[1] != embed_dim:
        continue
    p = np.asarray(prosody_list_raw[i], dtype=np.float32)
    if p.ndim != 2 or p.shape[1] != PROSODY_DIM or p.shape[0] != arr.shape[0]:
        continue
    if not np.isfinite(p).all():
        continue
    n = min(arr.shape[0], MAX_SEQ_LEN)
    if n > 0:
        X[i, :n, :] = arr[:n, :]
        X_prosody[i, :n, :] = p[:n, :]
        key_padding_mask[i, :n] = False
        valid_modal_rows[i] = True

valid_rows = np.isfinite(Y).all(axis=1) & valid_modal_rows
X = X[valid_rows]
X_prosody = X_prosody[valid_rows]
key_padding_mask = key_padding_mask[valid_rows]
Y = Y[valid_rows]

if len(Y) == 0:
    raise ValueError("No valid rows after filtering. Check PKL export and label availability.")

idx = np.arange(len(Y))
train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)

X_train, X_val = X[train_idx], X[val_idx]
X_prosody_train, X_prosody_val = X_prosody[train_idx], X_prosody[val_idx]
mask_train, mask_val = key_padding_mask[train_idx], key_padding_mask[val_idx]
y_train, y_val = Y[train_idx], Y[val_idx]

# Normalize prosody using train split statistics from valid (unpadded) timesteps only.
train_valid_steps = ~mask_train
prosody_mean = X_prosody_train[train_valid_steps].mean(axis=0).astype(np.float32)
prosody_std = X_prosody_train[train_valid_steps].std(axis=0).astype(np.float32)
prosody_std = np.where(prosody_std < 1e-6, 1.0, prosody_std).astype(np.float32)

X_prosody_train_norm = X_prosody_train.copy()
X_prosody_val_norm = X_prosody_val.copy()
X_prosody_train_norm[train_valid_steps] = (X_prosody_train_norm[train_valid_steps] - prosody_mean) / prosody_std
X_prosody_val_norm[~mask_val] = (X_prosody_val_norm[~mask_val] - prosody_mean) / prosody_std

# Normalize targets using train split statistics.
y_mean = y_train.mean(axis=0).astype(np.float32)
y_std = y_train.std(axis=0).astype(np.float32)
y_std = np.where(y_std < 1e-6, 1.0, y_std).astype(np.float32)
y_train_norm = (y_train - y_mean) / y_std
y_val_norm = (y_val - y_mean) / y_std

print(f"Loaded {N} rows from PKL, using {len(Y)} valid rows after filtering.")
print(
    f"X_text {X.shape}, X_prosody {X_prosody.shape}, Y {Y.shape}, embed_dim={embed_dim}"
)
print("Applied train-only normalization to prosody and targets.")

# ── 2) Dataset (text + prosody modalities) ─────────────────────────────────────
class MultimodalSeqDataset(Dataset):
    def __init__(self, X_text, X_prosody, key_padding_mask, y):
        self.X_text = torch.tensor(X_text, dtype=torch.float32)
        self.X_prosody = torch.tensor(X_prosody, dtype=torch.float32)
        self.key_padding_mask = torch.tensor(key_padding_mask, dtype=torch.bool)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_text[idx],
            self.X_prosody[idx],
            self.key_padding_mask[idx],
            self.y[idx],
        )

train_ds = MultimodalSeqDataset(X_train, X_prosody_train_norm, mask_train, y_train_norm)
val_ds = MultimodalSeqDataset(X_val, X_prosody_val_norm, mask_val, y_val_norm)

BATCH_SIZE = 16
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ── 3) Model: separate encoders (text vs prosody) + fusion + head ───────────
class ModalitySeqEncoder(nn.Module):
    """Transformer over one modality's segment features -> CLS vector (d_model,)."""

    def __init__(
        self,
        input_dim,
        max_seq_len,
        d_model=128,
        nhead=4,
        num_layers=2,
        dropout=0.1,
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, max_seq_len + 1, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x, key_padding_mask):
        b, seq_len, _ = x.shape
        x = self.input_proj(x)
        cls = self.cls_token.expand(b, 1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed[:, : seq_len + 1, :]
        cls_mask = torch.zeros((b, 1), dtype=torch.bool, device=x.device)
        full_mask = torch.cat([cls_mask, key_padding_mask], dim=1)
        x = self.encoder(x, src_key_padding_mask=full_mask)
        return x[:, 0, :]


class TwoTokenFusion(nn.Module):
    """Self-attention over [text_vec, audio_vec] then mean-pool."""

    def __init__(self, d_model, nhead, dropout):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)

    def forward(self, text_vec, audio_vec):
        x = torch.stack([text_vec, audio_vec], dim=1)
        x = self.encoder(x)
        return x.mean(dim=1)


class MultimodalTransformerRegressor(nn.Module):
    def __init__(
        self,
        text_dim,
        prosody_dim,
        max_seq_len,
        num_targets,
        d_model=128,
        nhead=4,
        num_layers=2,
        dropout=0.1,
        fusion="concat",
    ):
        super().__init__()
        self.fusion = fusion
        self.text_encoder = ModalitySeqEncoder(
            text_dim, max_seq_len, d_model, nhead, num_layers, dropout
        )
        self.audio_encoder = ModalitySeqEncoder(
            prosody_dim, max_seq_len, d_model, nhead, num_layers, dropout
        )
        if fusion == "concat":
            self.fuse_proj = nn.Sequential(
                nn.Linear(2 * d_model, d_model),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            self.two_token_fusion = None
        elif fusion == "attention":
            self.two_token_fusion = TwoTokenFusion(d_model, nhead, dropout)
            self.fuse_proj = None
        else:
            raise ValueError('fusion must be "concat" or "attention"')

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_targets),
        )

    def forward(self, x_text, key_padding_mask, x_prosody):
        text_repr = self.text_encoder(x_text, key_padding_mask)
        audio_repr = self.audio_encoder(x_prosody, key_padding_mask)
        if self.fusion == "concat":
            fused = self.fuse_proj(torch.cat([text_repr, audio_repr], dim=-1))
        else:
            fused = self.two_token_fusion(text_repr, audio_repr)
        return self.head(fused)


# Tuned for better stability on small/medium datasets.
D_MODEL = 64
NUM_LAYERS = 1
DROPOUT = 0.10
WEIGHT_DECAY = 1e-4
FUSION = "concat"

EPOCHS = 40
MIN_EPOCHS_BEFORE_STOP = 15
PATIENCE = 10
MIN_DELTA = 1e-4

LR_BASE = 1e-4
MAX_LR = 5e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultimodalTransformerRegressor(
    text_dim=embed_dim,
    prosody_dim=PROSODY_DIM,
    max_seq_len=MAX_SEQ_LEN,
    num_targets=num_targets,
    d_model=D_MODEL,
    nhead=4,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    fusion=FUSION,
).to(device)
print(model)
print("Targets:", LABEL_COLS)
print("Fusion:", FUSION)

# ── 4) Training loop (SmoothL1 + OneCycle LR) ─────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_BASE, weight_decay=WEIGHT_DECAY)
loss_fn = nn.SmoothL1Loss(beta=1.0)
GRAD_CLIP_NORM = 1.0
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    epochs=EPOCHS,
    steps_per_epoch=max(1, len(train_dl)),
    pct_start=0.2,
    anneal_strategy="cos",
    div_factor=10.0,
    final_div_factor=100.0,
)


def run_epoch(dl, train=True):
    model.train(train)
    total_loss = 0.0

    with torch.set_grad_enabled(train):
        for X_batch, X_prosody_batch, mask_batch, y_batch in dl:
            X_batch = X_batch.to(device)
            X_prosody_batch = X_prosody_batch.to(device)
            mask_batch = mask_batch.to(device)
            y_batch = y_batch.to(device)

            preds = model(X_batch, mask_batch, X_prosody_batch)
            loss = loss_fn(preds, y_batch)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()
                scheduler.step()

            total_loss += loss.item() * len(y_batch)

    return total_loss / len(dl.dataset)


best_val_loss = float("inf")
best_epoch = -1
epochs_no_improve = 0
best_state = None

for epoch in range(EPOCHS):
    train_loss = run_epoch(train_dl, train=True)
    val_loss = run_epoch(val_dl, train=False)

    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    elif epoch + 1 >= MIN_EPOCHS_BEFORE_STOP:
        epochs_no_improve += 1

    if (epoch + 1) % 5 == 0:
        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch+1:3d} | train SmoothL1: {train_loss:.4f} | val SmoothL1: {val_loss:.4f} | lr: {current_lr:.2e}"
        )

    if epoch + 1 >= MIN_EPOCHS_BEFORE_STOP and epochs_no_improve >= PATIENCE:
        print(
            f"Early stopping at epoch {epoch+1}. Best val SmoothL1 {best_val_loss:.4f} at epoch {best_epoch}."
        )
        break

if best_state is not None:
    device = next(model.parameters()).device
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

# Validation MAE in original label scale (de-normalized)
model.eval()
all_preds_norm, all_true_norm = [], []
with torch.no_grad():
    for X_batch, X_prosody_batch, mask_batch, y_batch in val_dl:
        X_batch = X_batch.to(device)
        X_prosody_batch = X_prosody_batch.to(device)
        mask_batch = mask_batch.to(device)
        preds_norm = model(X_batch, mask_batch, X_prosody_batch)
        all_preds_norm.append(preds_norm.detach().cpu().numpy())
        all_true_norm.append(y_batch.detach().cpu().numpy())

preds_norm = np.vstack(all_preds_norm)
true_norm = np.vstack(all_true_norm)
preds = preds_norm * y_std + y_mean
true = true_norm * y_std + y_mean
mae_per_target = np.mean(np.abs(preds - true), axis=0)

print("\nValidation MAE (original scale):")
for label, mae in zip(LABEL_COLS, mae_per_target):
    print(f"  {label:22s}: {mae:.4f}")
print(f"Mean MAE across targets: {mae_per_target.mean():.4f}")

Loaded 2011 rows from PKL, using 1996 valid rows after filtering.
X_text (1996, 64, 384), X_prosody (1996, 64, 13), Y (1996, 9), embed_dim=384
Applied train-only normalization to prosody and targets.
MultimodalTransformerRegressor(
  (text_encoder): ModalitySeqEncoder(
    (input_proj): Linear(in_features=384, out_features=64, bias=True)
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0): TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
          (linear1): Linear(in_features=64, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=256, out_features=64, bias=True)
          (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (

In [18]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error

model.eval()
all_preds_norm, all_true_norm = [], []
with torch.no_grad():
    for X_batch, X_prosody_batch, mask_batch, y_batch in val_dl:
        X_batch = X_batch.to(device)
        X_prosody_batch = X_prosody_batch.to(device)
        mask_batch = mask_batch.to(device)
        preds_norm = model(X_batch, mask_batch, X_prosody_batch)
        all_preds_norm.append(preds_norm.detach().cpu().numpy())
        all_true_norm.append(y_batch.detach().cpu().numpy())

preds_norm = np.vstack(all_preds_norm)
true_norm  = np.vstack(all_true_norm)
preds = preds_norm * y_std + y_mean
true  = true_norm  * y_std + y_mean

mae_per_target  = np.mean(np.abs(preds - true), axis=0)
rmse_per_target = np.sqrt(np.mean((preds - true) ** 2, axis=0))
r2_per_target   = np.array([r2_score(true[:, i], preds[:, i]) for i in range(num_targets)])
mape_per_target = np.array([
    mean_absolute_percentage_error(true[:, i], preds[:, i])
    for i in range(num_targets)
])

col_w = 24
print("\n" + "=" * 75)
print(f"{'VALIDATION METRICS (original scale)':^75}")
print("=" * 75)
print(f"{'Target':<{col_w}} {'MAE':>8} {'RMSE':>8} {'R²':>8} {'MAPE':>8}")
print("-" * 75)
for i, label in enumerate(LABEL_COLS):
    print(
        f"{label:<{col_w}} "
        f"{mae_per_target[i]:>8.4f} "
        f"{rmse_per_target[i]:>8.4f} "
        f"{r2_per_target[i]:>8.4f} "
        f"{mape_per_target[i]:>8.4f}"
    )
print("-" * 75)
print(
    f"{'MEAN':<{col_w}} "
    f"{mae_per_target.mean():>8.4f} "
    f"{rmse_per_target.mean():>8.4f} "
    f"{r2_per_target.mean():>8.4f} "
    f"{mape_per_target.mean():>8.4f}"
)
print("=" * 75)
print(f"\nBest val SmoothL1: {best_val_loss:.4f}  (epoch {best_epoch})")
print(f"Val samples: {len(true)}  |  Targets: {num_targets}")


                    VALIDATION METRICS (original scale)                    
Target                        MAE     RMSE       R²     MAPE
---------------------------------------------------------------------------
interview_score            0.6576   1.0126   0.1520   4.4707
overall_personality        0.6528   0.9838   0.1460   2.7775
answer_score               0.6993   1.1264   0.1262   1.8778
speaking_skills            0.7380   1.1745   0.1630   2.7109
agreeableness              0.6927   1.2136   0.1035   2.1454
conscientiousness          0.6064   0.9390   0.0934   6.2024
neuroticism                0.3497   0.4454   0.0328   2.6403
openness                   0.6707   1.0220   0.1455   2.3777
confidence_score           0.6396   0.9419   0.1464   3.6972
---------------------------------------------------------------------------
MEAN                       0.6341   0.9844   0.1232   3.2111

Best val SmoothL1: 0.2947  (epoch 4)
Val samples: 400  |  Targets: 9


In [19]:
# Save model checkpoints for server inference and normalized-training reproducibility

# Backward-compatible: weights only
torch.save(model.state_dict(), "multimodal_transformer_audio_regressor.pth")

# Full checkpoint: weights + normalization stats + config
checkpoint = {
    "state_dict": model.state_dict(),
    "label_mean": y_mean,
    "label_std": y_std,
    "prosody_mean": prosody_mean,
    "prosody_std": prosody_std,
    "config": {
        "text_dim": embed_dim,
        "prosody_dim": PROSODY_DIM,
        "max_seq_len": MAX_SEQ_LEN,
        "num_targets": num_targets,
        "d_model": D_MODEL,
        "nhead": 4,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "fusion": FUSION,
    },
}
torch.save(checkpoint, "multimodal_transformer_audio_regressor_with_norms.pth")

print("Saved:")
print("  multimodal_transformer_audio_regressor.pth")
print("  multimodal_transformer_audio_regressor_with_norms.pth")

Saved:
  multimodal_transformer_audio_regressor.pth
  multimodal_transformer_audio_regressor_with_norms.pth
